In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', 'ultralytics', 'pyyaml', '-q'])

from pathlib import Path
import os, shutil, yaml, random
print("Done!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Done!


In [ ]:
# ================================================================
# YOUR GOOGLE DRIVE PATH
# ================================================================
DRIVE_BASE   = Path("/content/drive/MyDrive/yolo_project work/datasets")
WORK_PATH    = Path("/content/datasets")
OUTPUT_DIR   = Path("/content/merged_dataset")
TRAIN_OUTPUT = Path("/content/drive/MyDrive/yolo_project/training_runs")

# ================================================================
# MASTER CLASS LIST — 7 CLASSES NOW INCLUDING VEHICLE
# Adding vehicle class fixes rickshaw being detected as vendor
# ================================================================
# 0 = barricade
# 1 = pothole
# 2 = illegal_parking
# 3 = street_vendor
# 4 = cart
# 5 = garbage
# 6 = vehicle  ← NEW: catches rickshaws, cars, trucks correctly
# ================================================================
CLASS_NAMES = [
    "barricade",
    "pothole",
    "illegal_parking",
    "street_vendor",
    "cart",
    "garbage",
    "vehicle",
]

# ================================================================
# DATASETS — exact folder names as in your Google Drive
# ================================================================
DATASETS = {
    "barricade": {
        "folder": "barricade.v1i.yolov5pytorch",
        "remap":  {0: 0, 1: 0, 2: 0},
    },
    "pothole_a": {
        "folder": "Pothole Detection.v1i.yolov8",
        "remap":  {0: 1},
    },
    "pothole_b": {
        "folder": "Potholes scanning.v1i.yolov8",
        "remap":  {0: 1},
    },
    "illegal_parking": {
        "folder": "illegal parking.v2i.yolov8",
        "remap":  {0: 2},
    },
    "street_vendor": {
        "folder": "street vendors.v1i.yolov8",
        "remap":  {0: 3},
    },
    "cart": {
        "folder": "Street Vendors India.v2i.yolov8",
        "remap":  {0: 4},
    },
    "garbage": {
        "folder": "d2.v1i.yolov8",
        # bin→garbage, garbage→garbage, pavement→skip, road→skip
        "remap":  {0: 5, 1: 5, 2: -1, 3: -1},
    },
    # -------------------------------------------------------
    # VEHICLE DATASET — download this from Roboflow
    # Search "vehicle detection" on universe.roboflow.com
    # Download as YOLOv8, rename folder to exactly this name
    # -------------------------------------------------------
    "vehicle": {
        "folder": "vehicle detection.v1i.yolov8",
        "remap":  {0: 6},
    },
}

# Scan Drive to verify folders
print("Checking dataset folders...")
print("=" * 55)
any_missing = False
for ds_name, cfg in DATASETS.items():
    path = DRIVE_BASE / cfg["folder"]
    if path.is_dir():
        total = sum(1 for p in path.rglob("*")
                    if p.suffix.lower() in {".jpg",".jpeg",".png"})
        print(f"  FOUND : '{cfg['folder']}' → {total} images")
    else:
        print(f"  MISSING: '{cfg['folder']}'")
        any_missing = True

if any_missing:
    print("\nSome folders missing — update names above or skip those datasets")
else:
    print("\nAll folders found! Proceed to Cell 3.")

Checking dataset folders...
  MISSING: 'barricade.v1i.yolov5pytorch'
  FOUND : 'Pothole Detection.v1i.yolov8' → 1879 images
  FOUND : 'Potholes scanning.v1i.yolov8' → 555 images
  FOUND : 'illegal parking.v2i.yolov8' → 811 images
  FOUND : 'street vendors.v1i.yolov8' → 495 images
  FOUND : 'Street Vendors India.v2i.yolov8' → 27 images
  FOUND : 'd2.v1i.yolov8' → 473 images
  MISSING: 'vehicle detection.v1i.yolov8'

Some folders missing — update names above or skip those datasets


In [ ]:
for path in (WORK_PATH, OUTPUT_DIR):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

print("Copying datasets from Drive to Colab...")
print("=" * 55)

for ds_name, cfg in DATASETS.items():
    src = DRIVE_BASE / cfg["folder"]
    dst = WORK_PATH / ds_name

    if not src.is_dir():
        print(f"  SKIPPING {ds_name} — folder not found")
        continue

    shutil.copytree(src, dst)
    total = sum(1 for p in dst.rglob("*")
                if p.suffix.lower() in {".jpg",".jpeg",".png"})
    print(f"  {ds_name}: {total} images")

print("=" * 55)
print("Done!")

Copying datasets from Drive to Colab...
  SKIPPING barricade — folder not found
  pothole_a: 1879 images
  pothole_b: 555 images
  illegal_parking: 811 images
  street_vendor: 495 images
  cart: 27 images
  garbage: 473 images
  SKIPPING vehicle — folder not found
Done!


In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}

def get_split(image_path):
    parts = {p.lower() for p in Path(image_path).parts}
    if "test"  in parts: return "test"
    if "valid" in parts or "val" in parts: return "valid"
    return "train"

def find_label(image_path):
    p = Path(image_path)
    c1 = p.with_suffix(".txt")
    if c1.exists(): return c1
    for i, part in enumerate(p.parts):
        if part.lower() == "images":
            parts = list(p.parts)
            parts[i] = "labels"
            c2 = Path(*parts).with_suffix(".txt")
            if c2.exists(): return c2
    return None

def remap_label(src, dst, remap):
    try:
        lines = Path(src).read_text(encoding="utf-8").splitlines()
    except:
        Path(dst).write_text("")
        return
    new_lines = []
    for line in lines:
        vals = line.strip().split()
        if len(vals) < 5: continue
        try:
            old_cls = int(vals[0])
            if old_cls not in remap or remap[old_cls] == -1:
                continue
            vals[0] = str(remap[old_cls])
            coords = [float(v) for v in vals[1:]]
            if len(coords) == 4:
                new_lines.append(" ".join(vals) + "\n")
            elif len(coords) >= 6 and len(coords) % 2 == 0:
                xs = coords[0::2]; ys = coords[1::2]
                xc=(min(xs)+max(xs))/2; yc=(min(ys)+max(ys))/2
                w=max(xs)-min(xs);     h=max(ys)-min(ys)
                new_lines.append(
                    f"{vals[0]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
        except: continue
    Path(dst).write_text("".join(new_lines), encoding="utf-8")

for split in ["train","valid","test"]:
    (OUTPUT_DIR/split/"images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR/split/"labels").mkdir(parents=True, exist_ok=True)

print("Merging datasets...")
print("=" * 55)
total_all = 0

for ds_name, cfg in DATASETS.items():
    ds_path = WORK_PATH / ds_name
    if not ds_path.exists():
        print(f"  SKIPPING {ds_name}")
        continue

    count = 0
    for img_path in ds_path.rglob("*"):
        if img_path.suffix.lower() not in IMAGE_SUFFIXES: continue
        split    = get_split(img_path)
        new_name = f"{ds_name}_{img_path.name}"
        new_stem = f"{ds_name}_{img_path.stem}"

        shutil.copy2(img_path, OUTPUT_DIR/split/"images"/new_name)

        lbl_src = find_label(img_path)
        lbl_dst = OUTPUT_DIR/split/"labels"/(new_stem+".txt")
        if lbl_src:
            remap_label(lbl_src, lbl_dst, cfg["remap"])
        else:
            lbl_dst.write_text("")
        count += 1

    print(f"  {ds_name}: {count} images")
    total_all += count

print("=" * 55)
print(f"TOTAL: {total_all} images merged")

Merging datasets...
  SKIPPING barricade
  pothole_a: 1879 images
  pothole_b: 555 images
  illegal_parking: 811 images
  street_vendor: 495 images
  cart: 27 images
  garbage: 473 images
  SKIPPING vehicle
TOTAL: 4240 images merged


In [ ]:
train_imgs = list((OUTPUT_DIR/"train"/"images").iterdir())
valid_imgs = list((OUTPUT_DIR/"valid"/"images").iterdir())

if len(valid_imgs) < 50:
    print(f"Valid set has {len(valid_imgs)} images — auto-splitting 15% from train...")
    random.seed(42)
    random.shuffle(train_imgs)
    to_move = train_imgs[:max(50, int(len(train_imgs)*0.15))]
    for img in to_move:
        shutil.move(str(img),
            str(OUTPUT_DIR/"valid"/"images"/img.name))
        lbl = OUTPUT_DIR/"train"/"labels"/(img.stem+".txt")
        if lbl.exists():
            shutil.move(str(lbl),
                str(OUTPUT_DIR/"valid"/"labels"/lbl.name))

# Verify annotation counts per class
class_counts = {i: 0 for i in range(len(CLASS_NAMES))}
for lbl in OUTPUT_DIR.glob("*/labels/*.txt"):
    for line in lbl.read_text().splitlines():
        vals = line.strip().split()
        if len(vals) == 5:
            cid = int(vals[0])
            if cid in class_counts:
                class_counts[cid] += 1

print("Annotations per class:")
for i, name in enumerate(CLASS_NAMES):
    count  = class_counts[i]
    status = "OK" if count > 100 else \
             "LOW — may cause poor accuracy" if count > 0 else \
             "EMPTY — this class will not be detected!"
    print(f"  {i} {name:<20}: {count:>6} annotations  [{status}]")

# data.yaml
data_yaml = {
    "path":  str(OUTPUT_DIR),
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}
yaml_path = OUTPUT_DIR/"data.yaml"
yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False))

t = len(list((OUTPUT_DIR/"train"/"images").iterdir()))
v = len(list((OUTPUT_DIR/"valid"/"images").iterdir()))
print(f"\nDataset ready:  Train={t}  Valid={v}")
print("data.yaml created!")

Annotations per class:
  0 barricade           :      0 annotations  [EMPTY — this class will not be detected!]
  1 pothole             :   5065 annotations  [OK]
  2 illegal_parking     :    832 annotations  [OK]
  3 street_vendor       :    729 annotations  [OK]
  4 cart                :     62 annotations  [LOW — may cause poor accuracy]
  5 garbage             :    373 annotations  [OK]
  6 vehicle             :      0 annotations  [EMPTY — this class will not be detected!]

Dataset ready:  Train=3300  Valid=696
data.yaml created!


In [ ]:
import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU! Go to Runtime → Change runtime type → T4 GPU")

print(f"GPU: {torch.cuda.get_device_name(0)}")
torch.cuda.empty_cache()

gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
batch   = 16 if gpu_mem >= 14 else 8 if gpu_mem >= 8 else 4
print(f"Auto batch size: {batch}")

model = YOLO("yolov8s.pt")  # using 'small' not 'nano' — better accuracy

results = model.train(
    data     = str(OUTPUT_DIR/"data.yaml"),
    epochs   = 80,           # increased from 50
    imgsz    = 640,
    batch    = batch,
    device   = 0,
    workers  = 2,
    patience = 15,
    project  = str(TRAIN_OUTPUT),
    name     = "road_defect_v2",
    exist_ok = True,
    amp      = True,
    cache    = True,
    plots    = True,
    verbose  = True,
    # Augmentation to compensate for small datasets
    hsv_h    = 0.015,
    hsv_s    = 0.7,
    hsv_v    = 0.4,
    flipud   = 0.1,
    fliplr   = 0.5,
    mosaic   = 1.0,
    mixup    = 0.1,
)

print("TRAINING COMPLETE!")

GPU: Tesla T4
Auto batch size: 16
Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/merged_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=road_defect_v2, nbs=64, nms=False, opset

In [ ]:
import glob as glob_module

save_dest = TRAIN_OUTPUT / "saved_weights"
save_dest.mkdir(parents=True, exist_ok=True)

matches = glob_module.glob(
    str(TRAIN_OUTPUT / "road_defect_v2*/weights/best.pt"))

if matches:
    best = sorted(matches)[-1]
    shutil.copy(best, str(save_dest/"best.pt"))
    print(f"Model saved to: {save_dest}/best.pt")
else:
    print("Model not found — check training completed successfully")

In [ ]:
from ultralytics import YOLO
import cv2, json
import glob as glob_module
from pathlib import Path
import matplotlib.pyplot as plt
from google.colab import files

# ================================================================
# MKJI 1997 TABLES — published standard values
# These are NOT made up — directly from the handbook
# ================================================================

# Table 2-1: Base capacity Co (PCU/hr/lane)
MKJI_BASE_CAPACITY = {
    "urban_divided":   1650,
    "urban_undivided": 1500,
    "rural_divided":   1900,
    "rural_undivided": 1800,
}

# Table 2-2: Lane width correction factor fw
# lane_width_m : fw_value
MKJI_FW_TABLE = [
    (3.50, 1.00),
    (3.25, 0.97),
    (3.00, 0.93),
    (2.75, 0.88),
    (2.50, 0.81),
    (0.00, 0.81),   # minimum
]

# Table 2-3: PCU equivalent for heavy vehicles
MKJI_ET = {"flat": 1.3, "rolling": 1.5, "hilly": 2.0}

# Table 2-4: Directional split factor
MKJI_FSP = {0.50:1.00, 0.55:0.97, 0.60:0.94, 0.65:0.91, 0.70:0.88}

# Width reduction per defect — based on IRC/field studies
# Basis: IRC SP-41 and MKJI obstruction adjustment
# These represent how much of ONE lane width a defect occupies
DEFECT_WIDTH_REDUCTION = {
    # defect_name: (min_reduction_m, max_reduction_m, basis)
    "barricade":       (1.00, 3.50, "IRC SP-41: barricade = 1 full lane"),
    "pothole":         (0.30, 1.20, "Vehicles swerve 0.3-1.2m around pothole"),
    "illegal_parking": (2.00, 3.50, "Parked vehicle = full lane blocked"),
    "street_vendor":   (0.50, 1.50, "Vendor + customers = 0.5-1.5m"),
    "cart":            (0.60, 1.80, "Cart wider than vendor stall"),
    "garbage":         (0.20, 0.80, "Garbage heap = minor obstruction"),
    "vehicle":         (0.00, 0.00, "Moving vehicle = not an obstruction"),
}

def get_fw(lane_width_m):
    """Get fw factor from MKJI Table 2-2."""
    for width, fw in MKJI_FW_TABLE:
        if lane_width_m >= width:
            return fw
    return 0.81

def get_fsp(directional_split):
    """Get fsp from MKJI Table 2-4."""
    closest = min(MKJI_FSP.keys(),
                  key=lambda k: abs(k - directional_split))
    return MKJI_FSP[closest]

def calc_capacity(lane_width_m, num_lanes, road_type,
                  heavy_pct, terrain, dir_split):
    """Full MKJI capacity formula with all factors."""
    Co  = MKJI_BASE_CAPACITY.get(road_type, 1650)
    fw  = get_fw(lane_width_m)
    ET  = MKJI_ET.get(terrain, 1.3)
    fHV = 1.0 / (1 + (heavy_pct/100) * (ET - 1))
    fsp = get_fsp(dir_split)
    C   = Co * fw * fHV * fsp * num_lanes
    return round(C, 1), {"Co":Co, "fw":fw,
                          "fHV":round(fHV,4), "fsp":fsp}

def get_los(loss_pct):
    if   loss_pct <  5: return "A", "Free Flow"
    elif loss_pct < 15: return "B", "Reasonably Free"
    elif loss_pct < 30: return "C", "Stable Flow"
    elif loss_pct < 45: return "D", "Approaching Unstable"
    elif loss_pct < 65: return "E", "Unstable Flow"
    else:               return "F", "Breakdown / Forced Flow"

def get_suggestion(defect_name, count, loss_pct):
    """Generate actionable suggestion per defect type."""
    suggestions = {
        "barricade":
            f"Remove or relocate {count} barricade(s). "
            f"They are blocking {loss_pct:.1f}% capacity. "
            "Coordinate with local municipal authority for alternative arrangement.",
        "pothole":
            f"{count} pothole(s) causing {loss_pct:.1f}% capacity loss. "
            "Immediate patching required as per IRC 37. "
            "Use cold mix for emergency repair within 24 hours.",
        "illegal_parking":
            f"{count} illegally parked vehicle(s) reducing capacity by "
            f"{loss_pct:.1f}%. "
            "Deploy traffic police or install no-parking signs + barriers. "
            "Consider towing enforcement.",
        "street_vendor":
            f"{count} street vendor(s) causing {loss_pct:.1f}% capacity loss. "
            "Relocate to designated vending zones as per street vendor policy. "
            "Municipal hawking committee must be notified.",
        "cart":
            f"{count} cart(s) on road reducing capacity by {loss_pct:.1f}%. "
            "Regulate cart movement to off-peak hours. "
            "Designate loading/unloading zones away from carriageway.",
        "garbage":
            f"{count} garbage dump(s) causing {loss_pct:.1f}% obstruction. "
            "Contact municipal solid waste department for immediate clearance. "
            "Install dustbins and educate citizens.",
        "vehicle":
            "Moving vehicles detected — no capacity action needed.",
    }
    return suggestions.get(defect_name, "Investigate and clear obstruction.")

def analyse_road_complete(image_path, model_path, road_config):
    """
    Complete road analysis with user-defined road parameters.

    road_config = {
        total_width_m, num_lanes, road_type,
        heavy_vehicle_pct, terrain, directional_split
    }
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Cannot read image: {image_path}")

    img_h, img_w = img.shape[:2]

    # pixels per meter using user-provided road width
    px_per_m   = img_w / road_config["total_width_m"]
    num_lanes  = road_config["num_lanes"]
    lane_w     = road_config["total_width_m"] / num_lanes

    # Original capacity with no defects
    orig_cap, orig_factors = calc_capacity(
        lane_width_m = lane_w,
        num_lanes    = num_lanes,
        road_type    = road_config["road_type"],
        heavy_pct    = road_config["heavy_vehicle_pct"],
        terrain      = road_config["terrain"],
        dir_split    = road_config["directional_split"],
    )

    # Run YOLO detection with higher confidence to reduce false positives
    model   = YOLO(str(model_path))
    pred    = model.predict(str(image_path), conf=0.45, verbose=False)[0]
    boxes   = pred.boxes

    print("\n" + "="*65)
    print("   ROAD EFFICIENCY ANALYSIS REPORT — INDO-HCM / MKJI 1997")
    print("="*65)
    print(f"  Image         : {Path(image_path).name}  ({img_w}×{img_h} px)")
    print(f"  Road Type     : {road_config['road_type']}")
    print(f"  Total Width   : {road_config['total_width_m']} m  "
          f"(user defined)")
    print(f"  Lanes         : {num_lanes}  →  "
          f"Lane width = {lane_w:.2f} m")
    print(f"  Heavy Vehicles: {road_config['heavy_vehicle_pct']}%")
    print(f"  Terrain       : {road_config['terrain']}")
    print(f"\n  MKJI TABLE VALUES USED:")
    print(f"  Co  = {orig_factors['Co']} PCU/hr/lane  "
          f"(MKJI Table 2-1, {road_config['road_type']})")
    print(f"  fw  = {orig_factors['fw']}  "
          f"(MKJI Table 2-2, lane width {lane_w:.2f}m)")
    print(f"  fHV = {orig_factors['fHV']}  "
          f"(MKJI Eq. 2-3, heavy={road_config['heavy_vehicle_pct']}%, "
          f"terrain={road_config['terrain']})")
    print(f"  fsp = {orig_factors['fsp']}  "
          f"(MKJI Table 2-4, split={road_config['directional_split']})")
    print(f"\n  ORIGINAL CAPACITY = Co × fw × fHV × fsp × lanes")
    print(f"  = {orig_factors['Co']} × {orig_factors['fw']} × "
          f"{orig_factors['fHV']} × {orig_factors['fsp']} × {num_lanes}")
    print(f"  = {orig_cap} PCU/hr")

    # Per-defect analysis
    defect_data   = {}
    total_blocked = 0.0

    roadrunner_export = []  # for MATLAB RoadRunner

    if boxes is not None and len(boxes) > 0:
        print(f"\n  DETECTED DEFECTS (conf ≥ 0.45):")
        print(f"  {'-'*60}")

        for box in boxes:
            cls_id   = int(box.cls[0])
            conf     = float(box.conf[0])
            cls_name = model.names[cls_id]

            # Skip vehicles — they are not obstructions
            if cls_name == "vehicle":
                print(f"  {cls_name:<20} MOVING VEHICLE — skipped")
                continue

            # Pixel dimensions
            bx, by, bw_px, bh_px = [float(v) for v in box.xywh[0]]

            # Convert to meters
            real_w_m = bw_px / px_per_m
            real_h_m = bh_px / px_per_m

            # Position on road (from left edge, meters)
            pos_x_m  = (bx - bw_px/2) / px_per_m
            pos_y_m  = (by - bh_px/2) / px_per_m

            # Width reduction = actual detected width capped at lane width
            # (from MKJI obstruction reduction method)
            blocked_m = min(real_w_m, lane_w)
            total_blocked += blocked_m

            if cls_name not in defect_data:
                defect_data[cls_name] = {
                    "count":      0,
                    "blocked_m":  0.0,
                    "detections": [],
                }
            defect_data[cls_name]["count"]     += 1
            defect_data[cls_name]["blocked_m"] += blocked_m
            defect_data[cls_name]["detections"].append({
                "conf":     round(conf,   2),
                "width_m":  round(real_w_m,2),
                "height_m": round(real_h_m,2),
                "pos_x_m":  round(pos_x_m, 2),
                "pos_y_m":  round(pos_y_m, 2),
            })

            # RoadRunner export row
            roadrunner_export.append({
                "defect_type": cls_name,
                "pos_x_m":     round(pos_x_m, 3),
                "pos_y_m":     round(pos_y_m, 3),
                "width_m":     round(real_w_m, 3),
                "height_m":    round(real_h_m, 3),
                "conf":        round(conf, 3),
            })

            # Validate width against expected range
            min_w, max_w, basis = DEFECT_WIDTH_REDUCTION.get(
                cls_name, (0,99,""))
            flag = ""
            if real_w_m < min_w:
                flag = f"  [NOTE: smaller than expected {min_w}m]"
            elif real_w_m > max_w:
                flag = f"  [NOTE: larger than expected {max_w}m]"

            print(f"  {cls_name:<20} "
                  f"w={real_w_m:.2f}m  "
                  f"blocks={blocked_m:.2f}m  "
                  f"conf={conf:.2f}{flag}")

    # Cap total blockage at road width
    total_blocked   = min(total_blocked, road_config["total_width_m"])
    effective_width = max(road_config["total_width_m"] - total_blocked, 0.5)
    eff_lane_w      = effective_width / num_lanes

    # Reduced capacity
    red_cap, red_factors = calc_capacity(
        lane_width_m = eff_lane_w,
        num_lanes    = num_lanes,
        road_type    = road_config["road_type"],
        heavy_pct    = road_config["heavy_vehicle_pct"],
        terrain      = road_config["terrain"],
        dir_split    = road_config["directional_split"],
    )

    cap_loss     = orig_cap - red_cap
    cap_loss_pct = (cap_loss / orig_cap) * 100
    los, los_desc = get_los(cap_loss_pct)

    print(f"\n  CAPACITY REDUCTION CALCULATION:")
    print(f"  {'-'*60}")
    print(f"  Original lane width    : {lane_w:.2f} m")
    print(f"  Total blocked by defects: {total_blocked:.2f} m")
    print(f"  Effective lane width   : {eff_lane_w:.2f} m")
    print(f"  New fw (reduced)       : {red_factors['fw']}  "
          f"(was {orig_factors['fw']})")
    print(f"\n  Reduced Capacity = {orig_factors['Co']} × "
          f"{red_factors['fw']} × {red_factors['fHV']} × "
          f"{red_factors['fsp']} × {num_lanes}")
    print(f"  = {red_cap} PCU/hr")
    print(f"\n  Original Capacity : {orig_cap:.0f} PCU/hr")
    print(f"  Reduced Capacity  : {red_cap:.0f} PCU/hr")
    print(f"  Capacity Lost     : {cap_loss:.0f} PCU/hr  "
          f"({cap_loss_pct:.1f}%)")
    print(f"  Level of Service  : {los} — {los_desc}")

    # Per-defect breakdown
    print(f"\n  PER-DEFECT CAPACITY BREAKDOWN:")
    print(f"  {'-'*60}")
    per_defect_results = {}

    for dname, dinfo in defect_data.items():
        # Capacity loss from this defect alone
        blocked_this  = min(dinfo["blocked_m"], road_config["total_width_m"])
        eff_w_this    = max(road_config["total_width_m"]-blocked_this, 0.5)
        cap_this, _   = calc_capacity(
            eff_w_this/num_lanes, num_lanes,
            road_config["road_type"],
            road_config["heavy_vehicle_pct"],
            road_config["terrain"],
            road_config["directional_split"],
        )
        loss_this     = orig_cap - cap_this
        loss_pct_this = (loss_this / orig_cap) * 100

        per_defect_results[dname] = {
            "count":        dinfo["count"],
            "blocked_m":    round(dinfo["blocked_m"], 2),
            "capacity_loss_pcu": round(loss_this, 1),
            "capacity_loss_pct": round(loss_pct_this, 1),
            "suggestion":   get_suggestion(dname, dinfo["count"],
                                           loss_pct_this),
        }

        print(f"\n  {dname.upper()} ({dinfo['count']} detected)")
        print(f"    Blocked width   : {dinfo['blocked_m']:.2f} m")
        print(f"    Capacity loss   : {loss_this:.0f} PCU/hr "
              f"({loss_pct_this:.1f}%)")
        print(f"    Suggestion      : "
              f"{get_suggestion(dname, dinfo['count'], loss_pct_this)}")

    # VALIDATION TABLE
    print(f"\n  VALIDATION TABLE (cross-check with MKJI handbook):")
    print(f"  {'-'*60}")
    print(f"  {'Parameter':<30} {'Calculated':>12} {'MKJI Source':>20}")
    print(f"  {'-'*60}")
    print(f"  {'Co (base cap/lane PCU/hr)':<30} "
          f"{orig_factors['Co']:>12} {'Table 2-1':>20}")
    print(f"  {'fw (orig lane {:.2f}m)'.format(lane_w):<30} "
          f"{orig_factors['fw']:>12} {'Table 2-2':>20}")
    print(f"  {'fw (reduced lane {:.2f}m)'.format(eff_lane_w):<30} "
          f"{red_factors['fw']:>12} {'Table 2-2':>20}")
    print(f"  {'fHV (heavy={:.0f}%, terrain={})'.format(road_config['heavy_vehicle_pct'], road_config['terrain']):<30} "
          f"{orig_factors['fHV']:>12} {'Eq. 2-3':>20}")
    print(f"  {'fsp (split={})'.format(road_config['directional_split']):<30} "
          f"{orig_factors['fsp']:>12} {'Table 2-4':>20}")
    print(f"  {'Original C (PCU/hr)':<30} "
          f"{orig_cap:>12} {'C=Co×fw×fHV×fsp×n':>20}")
    print(f"  {'Reduced C (PCU/hr)':<30} "
          f"{red_cap:>12} {'C=Co×fw×fHV×fsp×n':>20}")

    print("="*65)

    # Build final result dict
    final_result = {
        "image":                   Path(image_path).name,
        "road_config":             road_config,
        "original_capacity_pcu_hr": orig_cap,
        "reduced_capacity_pcu_hr":  red_cap,
        "capacity_loss_pcu_hr":     round(cap_loss, 1),
        "capacity_loss_pct":        round(cap_loss_pct, 1),
        "level_of_service":         los,
        "effective_width_m":        round(effective_width, 2),
        "mkji_factors":             orig_factors,
        "per_defect":               per_defect_results,
        "roadrunner_obstacles":     roadrunner_export,
    }

    # Save JSON for website
    json_path = Path(image_path).stem + "_analysis.json"
    with open(json_path, "w") as f:
        json.dump(final_result, f, indent=2)
    print(f"\n  JSON saved: {json_path}  (use this for website)")

    # Save CSV for MATLAB RoadRunner
    csv_path = Path(image_path).stem + "_roadrunner.csv"
    with open(csv_path, "w") as f:
        f.write("defect_type,pos_x_m,pos_y_m,width_m,height_m,conf\n")
        for obs in roadrunner_export:
            f.write(f"{obs['defect_type']},{obs['pos_x_m']},"
                    f"{obs['pos_y_m']},{obs['width_m']},"
                    f"{obs['height_m']},{obs['conf']}\n")
    print(f"  CSV saved : {csv_path}  (import this in MATLAB RoadRunner)")

    return final_result

print("Analysis functions ready! Run Cell 9.")

In [ ]:
from google.colab import files as colab_files
import matplotlib.pyplot as plt
import glob as glob_module, cv2

# ================================================================
# USER INPUTS — CHANGE THESE FOR YOUR ACTUAL ROAD
# ================================================================
ROAD_CONFIG = {
    "total_width_m":     7.0,   # ← CHANGE: measure your road width
    "num_lanes":         2,     # ← CHANGE: count your lanes
    "road_type":         "urban_divided",
    # Options:           "urban_divided"    (median road, city)
    #                    "urban_undivided"  (no median, city)
    #                    "rural_divided"    (median, highway)
    #                    "rural_undivided"  (no median, highway)
    "heavy_vehicle_pct": 15.0,  # ← CHANGE: % of trucks/buses
    "terrain":           "flat",
    # Options:           "flat" / "rolling" / "hilly"
    "directional_split": 0.50,
    # 0.50 = equal both directions
    # 0.60 = 60% one way, 40% other
}
# ================================================================

# Find trained model
matches = glob_module.glob(
    str(TRAIN_OUTPUT / "road_defect_v2*/weights/best.pt"))
if not matches:
    # Try old model name
    matches = glob_module.glob(
        str(TRAIN_OUTPUT / "road_defect_model*/weights/best.pt"))
if not matches:
    raise FileNotFoundError(
        "Trained model not found!\n"
        f"Looking in: {TRAIN_OUTPUT}\n"
        "Make sure training completed successfully.")

model_path = sorted(matches)[-1]
print(f"Using model: {model_path}")
print(f"\nRoad configuration:")
for k, v in ROAD_CONFIG.items():
    print(f"  {k}: {v}")

print("\nUpload your road image...")
uploaded = colab_files.upload()

for filename in uploaded:
    # Show annotated detection image
    model   = YOLO(model_path)
    results = model.predict(source=filename, conf=0.45,
                            save=False, verbose=False)
    annotated = results[0].plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(15, 8))
    plt.imshow(annotated)
    plt.axis("off")
    plt.title(f"Detected Defects — {filename}", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Full analysis
    result = analyse_road_complete(
        image_path  = filename,
        model_path  = model_path,
        road_config = ROAD_CONFIG,
    )

    # Download JSON and CSV files
    print("\nDownloading result files...")
    stem = Path(filename).stem
    colab_files.download(stem + "_analysis.json")
    colab_files.download(stem + "_roadrunner.csv")